In [1]:
import zmq
import cloudpickle
import time
import traceback

context = zmq.Context()

socket = context.socket(zmq.REP)
socket.bind("tcp://*:5556")

In [2]:
from qick import *
from qick.averager_program import *
soc = QickSoc()
soccfg = soc
print("Config chargée")

Config chargée


In [ ]:
while True:
    job = cloudpickle.loads(socket.recv())
    if job["type"] == "get_config":  
        socket.send(cloudpickle.dumps(soccfg.get_cfg()))
        print("config donnée")
        continue
    
    if job["type"] == "STOP":#stop n'est pas encore implémenté
        socket.send(cloudpickle.dumps({"status": "stopped"}))
        print("arret")
        break
        
    prog_class = job["prog"]
    config = job["config"]
    nom = job["nom"]
    threshold=job["threshold"]
    print("Traitement :", nom)
    try:
        prog = prog_class(soccfg,config)
        if job["acquire_method"]=="acquire_decimated":
            result = prog.acquire_decimated(soc,progress=True)
        elif job["acquire_method"]=="acquire_round":
            result = prog.acquire_round(soc,progress=True)
        elif job["acquire_method"]=="acquire":
            if threshold is not None:
                result = prog.acquire(soc,threshold=threshold,progress=True)
            else:
                result = prog.acquire(soc,progress=True)
        elif job["acquire_method"]=="run_rounds":
            result = prog.run_rounds(soc, rounds=job["rounds"],threshold=threshold, progress=False)
        else:
            raise ValueError(
                f"Unknown acquire_method: {job['acquire_method']}"
            )
        response = {
            "status": "done",
            "result": result}

    except Exception as e:
        print("erreur :" + type(e).__name__)
        response = {
        "status": "failed",
        "error_type": type(e).__name__,
        "result": traceback.format_exc()}
    socket.send(cloudpickle.dumps(response))
    print(f"Traitement de {nom} terminé")

config donnée
Traitement : my_job
erreur :ValueError
Traitement de my_job terminé
